# Features a construir:
 1. price_real por transacción
 2. RFM por hogar (Recency, Frequency, Monetary)
 3. display_flag y mailer_flag (codificación correcta del EDA)
 4. Merge con demographics y product (BRAND, DEPARTMENT)


In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
RUTA = '/content/drive/MyDrive/UCM -master Data Science/TFM/data/dunnhumby/'

In [4]:

df = {
    'transactions':  pd.read_csv(RUTA + 'transaction_data.csv'),
    'campaigns':     pd.read_csv(RUTA + 'campaign_table.csv'),
    'campaign_desc': pd.read_csv(RUTA + 'campaign_desc.csv'),
    'coupons':       pd.read_csv(RUTA + 'coupon.csv'),
    'redemptions':   pd.read_csv(RUTA + 'coupon_redempt.csv'),
    'causal':        pd.read_csv(RUTA + 'causal_data.csv'),
    'products':      pd.read_csv(RUTA + 'product.csv'),
    'demographics':  pd.read_csv(RUTA + 'hh_demographic.csv'),
}



In [5]:
trans = df['transactions'].copy()

# ¿Qué columnas monetarias tenemos?
cols_monetarias = ['SALES_VALUE', 'RETAIL_DISC', 'COUPON_DISC', 'COUPON_MATCH_DISC']
print(trans[cols_monetarias].describe().round(2).to_string())

       SALES_VALUE  RETAIL_DISC  COUPON_DISC  COUPON_MATCH_DISC
count   2595732.00   2595732.00   2595732.00         2595732.00
mean          3.10        -0.54        -0.02              -0.00
std           4.18         1.25         0.22               0.04
min           0.00      -180.00       -55.93              -7.70
25%           1.29        -0.69         0.00               0.00
50%           2.00        -0.01         0.00               0.00
75%           3.49         0.00         0.00               0.00
max         840.00         3.99         0.00               0.00


In [6]:
trans['tiene_retail_disc'] = trans['RETAIL_DISC'] != 0
trans['tiene_coupon_disc'] = trans['COUPON_DISC'] != 0
trans['tiene_match_disc'] = trans['COUPON_MATCH_DISC'] != 0

trans[['tiene_retail_disc', 'tiene_coupon_disc', 'tiene_match_disc']].mean()

,0
tiene_retail_disc,0.502002
tiene_coupon_disc,0.014031
tiene_match_disc,0.006722


In [7]:
sin_descuento = ~(trans['tiene_retail_disc'] | trans['tiene_coupon_disc'] | trans['tiene_match_disc'])
sin_descuento.mean()

np.float64(0.4916905905540325)

In [8]:
ejemplo = trans[trans['RETAIL_DISC'] != 0].head(1)
ejemplo[['SALES_VALUE', 'RETAIL_DISC', 'COUPON_DISC', 'COUPON_MATCH_DISC', 'QUANTITY']]

,SALES_VALUE,RETAIL_DISC,COUPON_DISC,COUPON_MATCH_DISC,QUANTITY
0,1.39,-0.6,0.0,0.0,1


In [9]:
sv = ejemplo['SALES_VALUE'].values[0]
rd = ejemplo['RETAIL_DISC'].values[0]
qty = ejemplo['QUANTITY'].values[0]

sv - rd, sv / qty

(np.float64(1.9899999999999998), np.float64(1.39))

Miro una transacción concreta con descuento para entender cómo se
relacionan las columnas. Parece que SALES_VALUE ya es lo que el
cliente pagó (con el descuento incluido), porque si le resto
RETAIL_DISC (que es negativo) me da un número más alto, que sería
el precio "de etiqueta" sin rebaja. Y si divido SALES_VALUE entre
QUANTITY tengo el precio que pagó por unidad.


 Si RETAIL_DISC es negativo, hay dos precios posibles:
  
  - A) Precio PAGADO    = SALES_VALUE / QUANTITY
     (lo que el cliente realmente pagó por unidad)

   - B) Precio ESTANTERÍA = (SALES_VALUE - RETAIL_DISC) / QUANTITY
      (el precio antes de descuento ,el de la etiqueta)

 Para Price Elasticity necesitamos los dos:
   - Precio estantería = la "decisión" del retailer
   - Precio pagado = lo que realmente afecta al comportamiento del cliente

In [10]:
#5 transacciones con descuento
ejemplos = trans[trans['RETAIL_DISC'] != 0].head(5).copy()

ejemplos['pagado'] = ejemplos['SALES_VALUE'] / ejemplos['QUANTITY']
ejemplos['estanteria'] = (ejemplos['SALES_VALUE'] - ejemplos['RETAIL_DISC']) / ejemplos['QUANTITY']
ejemplos['ahorro_pct'] = (1 - ejemplos['pagado'] / ejemplos['estanteria']) * 100

ejemplos[['SALES_VALUE', 'RETAIL_DISC', 'QUANTITY', 'pagado', 'estanteria', 'ahorro_pct']]

,SALES_VALUE,RETAIL_DISC,QUANTITY,pagado,estanteria,ahorro_pct
0,1.39,-0.60,1,1.39,1.99,30.150754
2,0.99,-0.30,1,0.99,1.29,23.255814
4,1.50,-0.39,1,1.50,1.89,20.634921
5,1.98,-0.60,2,0.99,1.29,23.255814
6,1.57,-0.68,1,1.57,2.25,30.222222


In [11]:
#5 transacciones sin descuento
ejemplos_sin = trans[trans['RETAIL_DISC'] == 0].head(5).copy()

ejemplos_sin['pagado'] = ejemplos_sin['SALES_VALUE'] / ejemplos_sin['QUANTITY']
ejemplos_sin['estanteria'] = (ejemplos_sin['SALES_VALUE'] - ejemplos_sin['RETAIL_DISC']) / ejemplos_sin['QUANTITY']
ejemplos_sin['ahorro_pct'] = (1 - ejemplos_sin['pagado'] / ejemplos_sin['estanteria']) * 100

ejemplos_sin[['SALES_VALUE', 'RETAIL_DISC', 'QUANTITY', 'pagado', 'estanteria', 'ahorro_pct']]

,SALES_VALUE,RETAIL_DISC,QUANTITY,pagado,estanteria,ahorro_pct
1,0.82,0.0,1,0.82,0.82,0.0
3,1.21,0.0,1,1.21,1.21,0.0
8,1.89,0.0,1,1.89,1.89,0.0
11,2.19,0.0,1,2.19,2.19,0.0
13,3.09,0.0,1,3.09,3.09,0.0


Comparo estas 5 transacciones sin descuento con las 5 anteriores
con descuento. Aquí pagado y estanteria salen iguales, como cabía
esperar si no hay descuento. Esto confirma que la lógica es
correcta: cuando no hay RETAIL_DISC, el precio pagado y el precio
de estantería coinciden, y cuando sí lo hay, la diferencia refleja
exactamente el ahorro. Mientras que con descuento los ahorros son del 20-30%.

#feature_Engineering  de precio





 - SALES_VALUE ya incluye descuentos aplicados
 - RETAIL_DISC se almacena como valor negativo
 - Sin descuento: precio pagado = precio estantería
 - Con descuento: ahorro típico 20-30%



  Se generan tres Creo las tres columnas de por transacción:
  - PRICE_PAID:     lo que el cliente pagó input para CLV (valor real gastado por cliente)
  - PRICE_SHELF:   precio de estantería input para Price Elasticity (decisión del retailer)
  - DISCOUNT_PCT:  % de descuento aplicado input para Promo Mechanic (intensidad del descuento)

In [12]:
trans['PRICE_PAID'] = trans['SALES_VALUE'] / trans['QUANTITY']
trans['PRICE_SHELF'] = (trans['SALES_VALUE'] - trans['RETAIL_DISC']) / trans['QUANTITY']
trans['DISCOUNT_PCT'] = np.where(
    trans['PRICE_SHELF'] > 0,
    1 - (trans['PRICE_PAID'] / trans['PRICE_SHELF']),
    0
)

In [13]:
trans[['PRICE_PAID', 'PRICE_SHELF', 'DISCOUNT_PCT']].describe()

/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,PRICE_PAID,PRICE_SHELF,DISCOUNT_PCT
count,2581333.00,2581350.00,2.595665e+06
mean,inf,NaN,1.335838e-01
std,NaN,NaN,1.703254e-01
min,0.00,-inf,-1.040000e+00
25%,1.00,1.29,0.000000e+00
50%,1.92,2.19,9.082652e-04
75%,2.99,3.49,2.471910e-01
max,inf,inf,1.000000e+00


Hay problemas: inf y NaN significan que hay filas con QUANTITY = 0 (división por cero) y posiblemente cantidades negativas (devoluciones).

Reviso si hay valores raros: precios negativos, en cero, o descuentos imposibles

In [14]:
#compruebo los casos raros
(trans['PRICE_PAID'] < 0).sum(), (trans['PRICE_PAID'] == 0).sum(), (trans['DISCOUNT_PCT'] > 1).sum()

(np.int64(0), np.int64(4451), np.int64(0))


# ¿Qué pasa con las filas que generan inf y NaN?


En la celda anterior me salieron valores infinitos y nulos en
PRICE_PAID y PRICE_SHELF. Sospecho que viene de QUANTITY=0, porque
estoy dividiendo por esa columna. Voy a mirar cuántas filas tienen
QUANTITY en 0 o negativo, y también si hay filas con SALES_VALUE=0,
para entender si son errores o algo con sentido (como devoluciones).

In [15]:
trans['QUANTITY'].describe()

,QUANTITY
count,2.595732e+06
mean,1.004286e+02
std,1.153436e+03
min,0.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,8.963800e+04


In [16]:
(trans['QUANTITY'] == 0).sum(), (trans['QUANTITY'] < 0).sum(), trans['QUANTITY'].isna().sum()

(np.int64(14466), np.int64(0), np.int64(0))

In [17]:
cols = ['household_key', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'RETAIL_DISC', 'COUPON_DISC']
trans[trans['QUANTITY'] == 0][cols].head()

,household_key,PRODUCT_ID,QUANTITY,SALES_VALUE,RETAIL_DISC,COUPON_DISC
97,744,5978648,0,0.0,0.0,0.0
128,1287,5978648,0,0.0,0.0,0.0
249,2305,5978656,0,0.0,0.0,-1.0
293,271,5978656,0,0.0,0.0,-2.0
694,315,957951,0,0.0,0.0,0.0


In [18]:
trans[trans['SALES_VALUE'] == 0][cols].head()

,household_key,PRODUCT_ID,QUANTITY,SALES_VALUE,RETAIL_DISC,COUPON_DISC
97,744,5978648,0,0.0,0.0,0.0
128,1287,5978648,0,0.0,0.0,0.0
249,2305,5978656,0,0.0,0.0,-1.0
293,271,5978656,0,0.0,0.0,-2.0
694,315,957951,0,0.0,0.0,0.0


In [19]:
problematicas = (trans['QUANTITY'] <= 0) | (trans['SALES_VALUE'] <= 0) | (trans['QUANTITY'].isna())
problematicas.sum(), problematicas.mean()

(np.int64(18917), np.float64(0.007287732323675942))

Decido eliminar las filas con QUANTITY<=0 o SALES_VALUE<=0. Son
18.917 filas (0.73% del total), un impacto pequeño, y ya vi que no
son devoluciones sino registros sin compra real. Dejarlas metería
infinitos y nulos en las columnas de precio, así que las quito antes
de calcular nada.Las voy a eliminar porque no aportan información y me están
rompiendo el cálculo de precio por unidad.


In [20]:
#filtro y veo cuántas filas quité
filas_antes = len(trans)
trans = trans[(trans['QUANTITY'] > 0) & (trans['SALES_VALUE'] > 0)].copy()
filas_despues = len(trans)

filas_antes, filas_despues, filas_antes - filas_despues

(2595732, 2576815, 18917)

Ahora repito el cálculo de PRICE_PAID, PRICE_SHELF y DISCOUNT_PCT
sobre los datos ya filtrados, y compruebo que esta vez no salgan
infinitos ni nulos

In [21]:
#creo las columnas
trans['PRICE_PAID'] = trans['SALES_VALUE'] / trans['QUANTITY']
trans['PRICE_SHELF'] = (trans['SALES_VALUE'] - trans['RETAIL_DISC']) / trans['QUANTITY']
trans['DISCOUNT_PCT'] = np.where(
    trans['PRICE_SHELF'] > 0,
    1 - (trans['PRICE_PAID'] / trans['PRICE_SHELF']),
    0
)

In [22]:

trans[['PRICE_PAID', 'PRICE_SHELF', 'DISCOUNT_PCT']].describe()

,PRICE_PAID,PRICE_SHELF,DISCOUNT_PCT
count,2.576815e+06,2.576815e+06,2.576815e+06
mean,2.441160e+00,2.844360e+00,1.331065e-01
std,2.720931e+00,3.018827e+00,1.674105e-01
min,1.658746e-03,1.758606e-03,-1.040000e+00
25%,1.000000e+00,1.290000e+00,0.000000e+00
50%,1.940000e+00,2.190000e+00,2.710027e-02
75%,2.990000e+00,3.490000e+00,2.480000e-01
max,4.999900e+02,5.499900e+02,9.979960e-01


Ya no hay infinitos ni nulos. Los precios tienen sentido: mediana
de 1.94 pagado y 2.19   de estantería, con un descuento medio del
13%

In [23]:
#check no queden infinitos ni nulos
np.isinf(trans['PRICE_PAID']).sum(), np.isinf(trans['PRICE_SHELF']).sum(), trans[['PRICE_PAID','PRICE_SHELF','DISCOUNT_PCT']].isna().sum().sum()

(np.int64(0), np.int64(0), np.int64(0))

#Feature Engineering RFM

Pasamos a la segunda feature: RFM por hogar. Pero primero, ¿qué es RFM? es la forma más clásica de describir el comportamiento de un cliente con tres números:

- Recency : ¿hace cuánto compró por última vez? (un cliente que compró ayer es más valioso que uno que compró hace 6 meses)
- Frequency : ¿cuántas veces ha comprado? (un cliente que viene cada semana es más fiel que uno que vino una vez)
- Monetary : ¿cuánto dinero ha gastado en total? (un cliente que gasta $500 vale más que uno que gasta $20)

In [24]:
ultimo_dia = trans['DAY'].max()
ultimo_dia

711

In [25]:
rfm = trans.groupby('household_key').agg(
    recency    = ('DAY', lambda x: ultimo_dia - x.max()),
    frequency  = ('BASKET_ID', 'nunique'),
    monetary   = ('SALES_VALUE', 'sum'),
    n_dias     = ('DAY', lambda x: x.max() - x.min()),
    primera    = ('DAY', 'min'),
    ultima     = ('DAY', 'max'),
).reset_index()

rfm['monetary_per_visit'] = rfm['monetary'] / rfm['frequency']
rfm.head()

,household_key,recency,frequency,monetary,n_dias,primera,ultima,monetary_per_visit
0,1,5,85,4330.16,655,51,706,50.943059
1,2,43,45,1954.34,565,103,668,43.429778
2,3,8,47,2653.21,590,113,703,56.451277
3,4,84,30,1200.11,523,104,627,40.003667
4,5,8,40,779.06,618,85,703,19.476500


In [26]:
rfm[['recency', 'frequency', 'monetary', 'monetary_per_visit']].describe()

,recency,frequency,monetary,monetary_per_visit
count,2500.000000,2500.000000,2500.000000,2500.000000
mean,25.576000,110.215600,3222.977520,31.728286
std,62.791673,115.271006,3349.026493,19.116288
min,0.000000,1.000000,8.170000,2.387500
25%,1.000000,38.000000,970.740000,18.432270
50%,6.000000,78.000000,2157.750000,27.488567
75%,20.000000,142.000000,4413.320000,40.619277
max,657.000000,1298.000000,38319.790000,165.829310


Para calcular RFM por hogar necesito tres cosas: hace cuánto compró
por última vez (recency), cuántas veces ha comprado (frequency), y
cuánto gasta (monetary).

Para frequency uso BASKET_ID en vez de contar filas, porque si
alguien compra 15 productos en una sola visita al súper eso cuenta
como 1 visita, no como 15. Si contara filas de transacción estaría
mezclando "cuántas veces vino" con "cuántos productos distintos
compró", que son cosas distintas.

Calculo también monetary_per_visit (gasto total dividido por número
de visitas) porque lo voy a necesitar más adelante para el modelo
Gamma-Gamma del notebook de CLV.

El hogar típico obtenido compró hace 6 días, ha hecho 78 visitas en los 2
años del dataset, y gasta unos 27.5$ por visita.

#Feature Engineer: flags promocionales desde causal_data

In [27]:
causal = df['causal'].copy()

causal['display_flag'] = (~causal['display'].isin(['0', 'A'])).astype(int)
causal['mailer_flag']  = (causal['mailer'] != '0').astype(int)
causal['any_promo'] = ((causal['display_flag'] == 1) | (causal['mailer_flag'] == 1)).astype(int)

Aplico aquí la codificación que validé en el EDA: display cuenta
como activo si el código no es "0" ni "A", y mailer cuenta como
activo si no es "0". Creo también un flag combinado por si en algún
momento necesito saber si hubo cualquier tipo de promo, sin
distinguir cuál.

Compruebo que las proporciones coincidan con lo que vi en el EDA:
sobre 40% con display activo, sobre 69% con mailer, y el % sin
ninguna promo cerca del 1.5% que había calculado antes. Si sale
muy distinto, algo estaría mal en el cruce de datos.

In [28]:
causal[['display_flag', 'mailer_flag', 'any_promo']].mean()

,0
display_flag,0.408699
mailer_flag,0.686456
any_promo,0.984713


Las proporciones coinciden: 40.9% con display, 68.6% con mailer,
98.5% con alguna promo activa (o sea, 1.5% sin ninguna, exactamente
lo que esperaba). La codificación está funcionando bien sobre el
dataset completo.

In [29]:
hogares_canjearon = set(df['redemptions']['household_key'].unique())

rfm['canjea_cupones'] = rfm['household_key'].isin(hogares_canjearon).astype(int)

In [30]:
master = rfm.copy()
len(master)

2500

In [31]:
'canjea_cupones' in master.columns

True

In [32]:
master = master.merge(df['demographics'], on='household_key', how='left')
master['AGE_DESC'].notna().sum()

np.int64(801)

In [33]:
trans_con_brand = trans.merge(df['products'][['PRODUCT_ID', 'BRAND']], on='PRODUCT_ID', how='left')

brand_mix = trans_con_brand.groupby(['household_key', 'BRAND'])['SALES_VALUE'].sum().unstack(fill_value=0).reset_index()
brand_mix['pct_national'] = brand_mix['National'] / (brand_mix['National'] + brand_mix['Private'])

master = master.merge(brand_mix[['household_key', 'pct_national']], on='household_key', how='left')
master['pct_national'].describe()

,pct_national
count,2500.000000
mean,0.727185
std,0.117094
min,0.019523
25%,0.657891
50%,0.737198
75%,0.810829
max,1.000000


In [34]:
precio_medio = trans.groupby('household_key')['PRICE_PAID'].mean().reset_index()
precio_medio.columns = ['household_key', 'avg_price_paid']

descuento_medio = trans.groupby('household_key')['DISCOUNT_PCT'].mean().reset_index()
descuento_medio.columns = ['household_key', 'avg_discount_pct']

master = master.merge(precio_medio, on='household_key', how='left')
master = master.merge(descuento_medio, on='household_key', how='left')
master.shape

(2500, 19)

In [35]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   household_key        2500 non-null   int64  
 1   recency              2500 non-null   int64  
 2   frequency            2500 non-null   int64  
 3   monetary             2500 non-null   float64
 4   n_dias               2500 non-null   int64  
 5   primera              2500 non-null   int64  
 6   ultima               2500 non-null   int64  
 7   monetary_per_visit   2500 non-null   float64
 8   canjea_cupones       2500 non-null   int64  
 9   AGE_DESC             801 non-null    object 
 10  MARITAL_STATUS_CODE  801 non-null    object 
 11  INCOME_DESC          801 non-null    object 
 12  HOMEOWNER_DESC       801 non-null    object 
 13  HH_COMP_DESC         801 non-null    object 
 14  HOUSEHOLD_SIZE_DESC  801 non-null    object 
 15  KID_CATEGORY_DESC    801 non-null    o

Uno todas las features en un solo DataFrame, una fila por hogar:
el RFM, los datos demográficos (solo
disponibles para el 32% de los hogares), el % de gasto en marca
National frente a Private, y el precio medio y descuento medio que
capta cada hogar.
Ahora me queda un maestro de 2.500 hogares con 22
columnas, listo para usar en los modelos.

In [36]:
# Guardar features
RUTA_OUTPUT = '/content/drive/MyDrive/UCM -master Data Science/TFM/data/processed/'

master.to_csv(RUTA_OUTPUT + 'master_hogares.csv', index=False)
trans.to_csv(RUTA_OUTPUT + 'transactions_clean.csv', index=False)

In [37]:
from google.colab import files

In [38]:
!jupyter nbconvert --to html "/content/drive/MyDrive/UCM -master Data Science/TFM/notebooks/01_FEATURE ENGINEERING.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/UCM -master Data Science/TFM/notebooks/01_FEATURE ENGINEERING.ipynb to html
[NbConvertApp] Writing 399870 bytes to /content/drive/MyDrive/UCM -master Data Science/TFM/notebooks/01_FEATURE ENGINEERING.html
